# [Chapter 12 — Two-Group Models, §12.3] Two-Group $SIR_I$ Dynamics: Setup and Numerical Solution

**Book:** *Essential Considerations for Modeling Epidemics* (Hyman/Qu/Xue, 2026)
**Chapter:** Chapter 12 — Two-Group Models
**Considerations developed:** 4 (force of infection vs from), 7 (heterogeneous-mixing balance)
**Estimated runtime:** ≤ 30 s

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/machyman/hyman2026essential/blob/main/python/notebooks/chapter_12_two_group/01_two_group_setup_and_dynamics.ipynb)


## What this notebook does

Builds a two-group $SIR_I$ model from first principles: starting from per-group force-of-infection expressions and the symmetry-balance constraint $c_{ij} N_i = c_{ji} N_j$ (Consideration~7). The notebook implements proportional mixing as the canonical example, integrates the resulting 6-ODE system, and shows that the higher-activity group drives the epidemic — a qualitative phenomenon invisible in any single-group reduction.

## Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..')))
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint
from shared import book_style, BOOK_COLORS
from shared.seeds import set_seed_chapter_12
from shared.verification import assert_within_tolerance
set_seed_chapter_12()
book_style()


## Mixing matrix and force of infection

Under proportional mixing,
$$c_{ij} = \frac{c_j N_j}{\sum_k c_k N_k},$$
which automatically satisfies the symmetry $c_{ij} c_i = c_{ji} c_j$ (Consideration~7). The force of infection on a susceptible in group $i$ is
$$\Lambda_i = \beta \, c_i \sum_j c_{ij} \frac{I_j}{N_j}.$$

In [ ]:
from shared.parameters import baseline_chapter_12
p = baseline_chapter_12()

c1, c2 = p['c1'], p['c2']
N1, N2 = p['N1'], p['N2']
beta, tau_R = p['beta'], p['tau_R']

# Proportional-mixing matrix
denom = c1*N1 + c2*N2
M = np.array([[c1*N1, c2*N2], [c1*N1, c2*N2]]) / denom
print('Proportional-mixing matrix M:')
print(M)
print()
# Symmetry check (Consideration 7)
lhs = M[0,1] * c1 * N1
rhs = M[1,0] * c2 * N2
print(f'Symmetry check c_12 c_1 N_1 = c_21 c_2 N_2:  {lhs:.4f} == {rhs:.4f}')
assert_within_tolerance(lhs, rhs, abs_tol=1e-9, label='proportional mixing satisfies symmetry')


## Two-group system

In [ ]:
def two_group_rhs(y, t, p, M):
    S1, I1, R1, S2, I2, R2 = y
    N1, N2 = p['N1'], p['N2']
    c1, c2 = p['c1'], p['c2']
    beta, tau_R = p['beta'], p['tau_R']
    Lam1 = beta * c1 * (M[0,0]*I1/N1 + M[0,1]*I2/N2)
    Lam2 = beta * c2 * (M[1,0]*I1/N1 + M[1,1]*I2/N2)
    return [-Lam1*S1, Lam1*S1 - I1/tau_R, I1/tau_R,
            -Lam2*S2, Lam2*S2 - I2/tau_R, I2/tau_R]

I0_each = 0.0001
y0 = [N1 - I0_each, I0_each, 0.0,
      N2 - I0_each, I0_each, 0.0]
t = np.linspace(p['t_start'], p['t_end'], p['n_points'])
sol = odeint(two_group_rhs, y0, t, args=(p, M))
S1, I1, R1, S2, I2, R2 = sol.T


## High-activity group drives the epidemic

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
axes[0].plot(t, I1/N1, color=BOOK_COLORS['primary'], lw=2)
axes[0].set_title(f'Group 1 (low activity, c={c1})')
axes[0].set_xlabel('time (days)')
axes[0].set_ylabel('infectious fraction within group')
axes[1].plot(t, I2/N2, color=BOOK_COLORS['secondary'], lw=2)
axes[1].set_title(f'Group 2 (high activity, c={c2})')
axes[1].set_xlabel('time (days)')
for a in axes: a.grid(True, alpha=0.3)
fig.suptitle('Group 2 sustains higher prevalence at peak')
fig.tight_layout()
plt.show()

att1 = float(R1[-1] + I1[-1])/N1
att2 = float(R2[-1] + I2[-1])/N2
print(f'Final attack rate, group 1: {att1:.3f}')
print(f'Final attack rate, group 2: {att2:.3f}')
assert att2 > att1, 'high-activity group should have higher attack rate'


## What's next

- Notebook 02 derives $\mathcal{R}_0$ for this system using the next-generation matrix.
- Chapter 13 specializes to bipartite structures (sexual transmission) where the off-diagonal mixing dominates.
- The asymmetry between groups motivates targeted intervention design — see book §9.6.